# ferrite-tpu on a real TPU

Runs the whole project on the TPU attached to this notebook: the test-suite (real Mosaic
lowering, no interpreter), the kernel benchmarks with roofline numbers, two short training
runs (Pallas kernels vs XLA), the serving benchmark and a profiler trace. Results land in
`ferrite-tpu/results/<device>/` and are zipped at the end so they can be committed to the repo.

**Kaggle:** Settings -> Accelerator -> *TPU VM v3-8* (or v5e-8). Needs phone verification and
internet enabled. **Colab:** Runtime -> Change runtime type -> TPU. Expect 15-25 minutes end to end.

**One rule:** this kernel never imports JAX. A TPU chip is owned by a single process, so every stage below runs as its own `python` process; importing JAX here would lock the shell cells out with `ABORTED: The TPU is already in use`.


In [ ]:
REPO_URL = "https://github.com/anxnd2211/ferrite-tpu.git"
import os, subprocess
if not os.path.isdir("ferrite-tpu"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ferrite-tpu"], check=True)
%pip install -q -U "jax[tpu]"
%pip install -q -e "./ferrite-tpu[dev]"


In [ ]:
import subprocess, sys
CHECK = r"""
import json, jax
from ferrite import platform
assert jax.default_backend() == "tpu", "no TPU attached: fix the accelerator setting and restart the runtime"
print(jax.__version__, jax.devices())
print(json.dumps(platform.describe(), indent=2))
"""
print(subprocess.run([sys.executable, "-c", CHECK], cwd="ferrite-tpu", capture_output=True, text=True, check=True).stdout)
DEV = subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0, 'benchmarks'); from _common import device_slug; print(device_slug())"],
                     cwd="ferrite-tpu", capture_output=True, text=True, check=True).stdout.strip()
RES = f"results/{DEV}"
print("results ->", RES)


## 1. Tests on real hardware
Same tests as CI, but the kernels now go through the Mosaic compiler and run on the TPU.

In [ ]:
!cd ferrite-tpu && python -m pytest -q -x -p no:cacheprovider

## 2. Kernel benchmarks + roofline
Each kernel against its XLA baseline (and JAX's own Pallas flash attention), with achieved TFLOP/s, GB/s and percent of the roofline for this chip.

In [ ]:
!cd ferrite-tpu && python benchmarks/bench_kernels.py --iters 20

## 3. Training: Pallas kernels vs XLA
13M-parameter decoder on byte-level TinyShakespeare, data-parallel over all cores. Same seed, same data order; compare step time, tokens/s and MFU.

In [ ]:
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention pallas --norm pallas --out {RES}/train_pallas.json --save-params checkpoints/pallas.npz
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention xla --norm xla --out {RES}/train_xla.json
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention pallas --norm pallas --linear pallas --out {RES}/train_pallas_all.json


## 4. Serving: AOT-compiled greedy decoding
TTFT, inter-token latency and tokens/s at several batch sizes, with Pallas vs XLA prefill, using the checkpoint from step 3.

In [ ]:
!cd ferrite-tpu && python benchmarks/bench_serve.py --params checkpoints/pallas.npz --seq 512 --batches 1,8,32 --prompt-len 100 --new-tokens 128

## 5. Profiler trace
Captures a device trace of the flash-attention kernel (forward + backward) next to the XLA baseline. Open the folder in TensorBoard (`tensorboard-plugin-profile`) or upload the `.xplane.pb` to https://ui.perfetto.dev to see each `tpu_custom_call` with its measured duration.

In [ ]:
!cd ferrite-tpu && python benchmarks/trace_attention.py --shape 4,8,2048,128 --steps 5

## 6. Report

In [ ]:
!cd ferrite-tpu && python benchmarks/make_report.py results > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"ferrite-tpu/{RES}/REPORT.md").read()))


In [ ]:
!cd ferrite-tpu && zip -qr ../ferrite_results.zip results traces && ls -la ../ferrite_results.zip
print("Download ferrite_results.zip (Output panel), unzip into the repo, commit results/<device>/ and traces/.")
